In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Regular vs Irregular 

This notebook aims to introduce the difference between regular and irregular time series, and the implications. 

- A **regular time series** is one where observations are recorded at consistent, evenly spaced time intervals (e.g., every hour, day, or month). This regularity simplifies analysis and is often assumed by statistical and machine learning models.
- An **irregular time series** has observations that occur at uneven or unpredictable intervals. This can complicate analysis, requiring additional steps such as resampling or interpolation to convert the data into a regular format suitable for many modeling techniques.

## Illustrative Example

Let's generate a signal, and sample at regular and irregular frequencies to show the difference. We shall use a sine wave, as this is a familiar signal.

In [ ]:
y = list(map(np.sin, np.linspace(0, 2*np.pi)))
x = range(len(y))

df = pd.DataFrame({"x": x, "y": y})
df.head()

In [ ]:
# create dummy sine wave data
x_full = np.linspace(0, 2 * np.pi, 1000)
y_full = np.sin(x_full)
df_full = pd.DataFrame({"y": y_full}, index=pd.date_range(start=pd.Timestamp("2025-01-01"), periods=len(x_full), freq="h"))
df_full.head()

In [ ]:
# regularly sample every 50th point
df_regular = df_full.iloc[::50]

# irregular sampling
df_irregular = df_full.sample(10)

# plot
fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True)

# Full sine wave
sns.lineplot(df_full, ax=axs[0], color="black", legend=False)
axs[0].set_title("Complete Sine wave")
axs[0].set_ylabel("Amplitude")

# Regularly sampled points
sns.scatterplot(data=df_regular, ax=axs[1], color="blue", legend=False)
axs[1].set_title("Regular Sampling (every 50 hours)")
axs[1].set_ylabel("Amplitude")

# Irregular sampling
sns.scatterplot(data=df_irregular, ax=axs[2], color="red", legend=False)
axs[2].set_title("Irregular Sampling (random timestamps)")
axs[2].set_ylabel("Amplitude")
axs[2].set_xlabel("Time")

plt.tight_layout()
plt.show()


This figure shows the impact of sampling structure on a simple periodic signal:

1. **True signal**: the underlying function sin(x), continuous and smooth.
2. **Regular sampling**: evenly spaced time points preserve the wave's structure.
3. **Irregular sampling**: randomly chosen time points obscure the true signal's shape, which affects modeling and forecasting.

This highlights why **time-based spacing is critical**. Most models assume regularity, and irregular data must be resampled or interpolated first.


## How to Check if a Time Series is Regular

1. **Plot it:** Visual inspection often reveals irregularities.This is evident above
2. **Infer frequency:** `pd.infer_freq()` tries to detect regular intervals, and returns `None` if it fails.
3. **Check time gaps:** Examine and plot the differences between consecutive timestamps.
4. **Count unique gaps:** If there’s only one unique time delta, it’s regular.

### Check Frequency with pandas

In [ ]:
freq = pd.infer_freq(df_regular.index)
print(freq)

In [ ]:
freq = pd.infer_freq(df_irregular.index)
print(freq)

### Analyse Time Gaps

In [ ]:
time_diffs_regular = df_regular.index.to_series().diff().dropna()
time_diffs_regular.value_counts()

In [ ]:
time_diffs_irregular = df_irregular.index.to_series().diff().dropna()
time_diffs_irregular.value_counts()

In [ ]:
fig, axs = plt.subplots(1, 2)
fig.suptitle("Histogram of Time Differences")

time_diffs_regular.dt.total_seconds().hist(bins=20, ax=axs[0])
axs[0].set_title("Regular")
axs[0].set_xlabel("Seconds between samples")

time_diffs_irregular.dt.total_seconds().hist(bins=20, ax=axs[1])
axs[1].set_title("Irregular")
axs[1].set_xlabel("Seconds between samples")

fig.tight_layout()
plt.show()

- **Single dominant time gap:**  
  If the differences between consecutive timestamps predominantly consist of a single, consistent interval, the time series can be considered **regularly spaced**

- **Multiple distinct time gaps:**  
  If there are multiple different intervals between consecutive timestamps, indicating inconsistency in sampling frequency, the time series is considered **irregularly spaced**

### Programmatic Regularity Check

In [ ]:
is_regular = time_diffs_regular.nunique() == 1
print("Regular" if is_regular else "Irregular")

In [ ]:
is_regular = time_diffs_irregular.nunique() == 1
print("Regular" if is_regular else "Irregular")

## Converting Irregular Time Series to Regular

Many time series models and analysis techniques require data to be evenly spaced in time. Converting an irregular time series to a regular one enables the use of these methods, simplifies handling of missing data, and makes it easier to visualise and interpret trends and patterns. Regularisation also helps ensure consistency and comparability across datasets.

### Reindexing

Reindexing creates a new, regular time index over the desired period and aligns the existing data to this index. Any timestamps in the new index that do not exist in the original data will result in missing values (`NaN`).

In [ ]:
target_freq = '1h'
regular_index = pd.date_range(start=df_irregular.index.min(), end=df_irregular.index.max(), freq=target_freq)
df_irregular_reindexed = df_irregular.reindex(regular_index)
df_irregular_reindexed.head()

In [ ]:
df_irregular_ffill = df_irregular_reindexed.ffill()
df_irregular_ffill.head()

In [ ]:
df_irregular_interp = df_irregular_reindexed.interpolate(method='time')
df_irregular_interp.head()

In [ ]:
# plot
fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True)

# Irregular sampling
sns.scatterplot(data=df_irregular, ax=axs[0], legend=False)
axs[0].set_title("Irregular Sampling")
axs[0].set_ylabel("Amplitude")
axs[0].set_xlabel("Time")

# Reindexed ffill
sns.scatterplot(data=df_irregular_ffill, ax=axs[1], legend=False)
axs[1].set_title("Reindexed (1h, ffill)")
axs[1].set_ylabel("Amplitude")

# Reindexed interpolated
sns.scatterplot(data=df_irregular_interp, ax=axs[2], legend=False)
axs[2].set_title("Reindexed (interpolated)")
axs[2].set_ylabel("Amplitude")

plt.tight_layout()
plt.show()

### Resampling

Resampling is the process of converting a time series from one frequency to another, either by upsampling (increasing the frequency, e.g., from daily to hourly) or downsampling (decreasing the frequency, e.g., from hourly to daily). This is typically done using aggregation methods (such as mean, sum, or count) to summarise data within each new time bin.

In [ ]:
# Upsample
hourly_data = df_irregular.resample('1h').mean()

# Fill missing data after upsampling by interpolation
hourly_data_filled = hourly_data.interpolate(method='time')
hourly_data_filled.head()

In [ ]:
# plot
fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Irregular sampling
sns.scatterplot(data=df_irregular, ax=axs[0], legend=False)
axs[0].set_title("Irregular Sampling")
axs[0].set_ylabel("Amplitude")
axs[0].set_xlabel("Time")

# Reindexed ffill
sns.scatterplot(data=hourly_data_filled, ax=axs[1], legend=False)
axs[1].set_title("Resampled (1h, interpolated)")
axs[1].set_ylabel("Amplitude")

plt.tight_layout()
plt.show()

We can see that using a target frequency of `1h` there is little / no difference between resampling and reindexing.

Let's repeat the above processes, but this time using a lower frequency.

In [ ]:
freq = '50h'

# Create a regular datetime index over the span of df_irregular
regular_index = pd.date_range(start=df_irregular.index.min(), end=df_irregular.index.max(), freq=freq)

# Reindex and interpolate
df_reindexed = df_irregular.reindex(regular_index)
df_reindexed_interp = df_reindexed.interpolate(method='time')

# Resample with aggregation (mean), then interpolate missing points
df_resampled = df_irregular.resample(freq).mean()
df_resampled_interp = df_resampled.interpolate(method='time')

# plot
fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True)

# Original irregular data
sns.scatterplot(data=df_irregular, x=df_irregular.index, y='y', ax=axs[0], color='black')
axs[0].set_title('Original Irregular Data')
axs[0].set_ylabel('Amplitude')

# Reindexed + interpolated
sns.lineplot(data=df_reindexed_interp, x=df_reindexed_interp.index, y='y', ax=axs[1], color='blue')
axs[1].set_title('Reindexed + Interpolated')
axs[1].set_ylabel('Amplitude')

# Resampled + interpolated
sns.lineplot(data=df_resampled_interp, x=df_resampled_interp.index, y='y', ax=axs[2], color='red')
axs[2].set_title('Resampled (Mean) + Interpolated')
axs[2].set_ylabel('Amplitude')
axs[2].set_xlabel('Time')

plt.tight_layout()
plt.show()

### Comparison

- At high frequencies (e.g. `'1H'`), both approaches produce very similar results since the data is dense and gaps are small.
- At lower frequencies (e.g. `'50H'`), differences become significant:
  - **Reindexing** may discard large portions of the data if there are no points near the target timestamps. This leads to flatter interpolated segments and loss of variability.
  - **Resampling** better preserves the overall signal by aggregating within each time bin. It smooths the data but retains more structure and avoids over-interpolating over large gaps.